# Rekurentní neuronové sítě

V předchozím modulu jsme pokryli bohaté sémantické reprezentace textu. Architektura, kterou používáme, zachycuje agregovaný význam slov ve větě, ale nezohledňuje **pořadí** slov, protože agregační operace následující po embeddingu odstraní tuto informaci z původního textu. Protože tyto modely nejsou schopny reprezentovat pořadí slov, nemohou řešit složitější nebo nejednoznačné úkoly, jako je generování textu nebo odpovídání na otázky.

Pro zachycení významu sekvence textu použijeme neuronovou síť s architekturou nazývanou **rekurentní neuronová síť**, nebo RNN. Při použití RNN procházíme naši větu sítí po jednom tokenu a síť produkuje nějaký **stav**, který pak předáme síti společně s dalším tokenem.

![Obrázek ukazující příklad generování rekurentní neuronové sítě.](../../../../../translated_images/cs/rnn.27f5c29c53d727b5.webp)

Vzhledem k vstupní sekvenci tokenů $X_0,\dots,X_n$ RNN vytvoří sekvenci neuronových bloků a tuto sekvenci trénuje end-to-end pomocí zpětné propagace. Každý blok sítě přijímá pár $(X_i,S_i)$ jako vstup a produkuje $S_{i+1}$ jako výsledek. Konečný stav $S_n$ nebo výstup $Y_n$ jde do lineárního klasifikátoru, který vytvoří výsledek. Všechny bloky sítě sdílejí stejné váhy a jsou trénovány end-to-end pomocí jedné zpětné propagace.

> Obrázek nahoře ukazuje rekurentní neuronovou síť v rozvinuté podobě (nalevo) a v kompaktnější rekurentní reprezentaci (napravo). Je důležité si uvědomit, že všechny RNN buňky mají stejné **sdílené váhy**.

Protože stavové vektory $S_0,\dots,S_n$ jsou předávány sítí, RNN je schopna se naučit sekvenční závislosti mezi slovy. Například když se ve sekvenci vyskytne slovo *not* (ne), může se naučit negovat určité prvky ve stavovém vektoru.

Uvnitř každé RNN buňky jsou dvě matice vah: $W_H$ a $W_I$ a bias $b$. V každém RNN kroku je při vstupu $X_i$ a stavovém vstupu $S_i$ výstupní stav vypočten jako $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, kde $f$ je aktivační funkce (často $\tanh$).

> Pro problémy jako generování textu (které pokryjeme v další jednotce) nebo strojový překlad také chceme získat nějakou výstupní hodnotu v každém RNN kroku. V tomto případě existuje další matice $W_O$ a výstup se počítá jako $Y_i=f(W_O\times S_i+b_O)$.

Podívejme se, jak nám rekurentní neuronové sítě mohou pomoci klasifikovat náš dataset zpráv.

> Pro sandbox prostředí je potřeba spustit následující buňku, aby byla nainstalována požadovaná knihovna a aby byla data přednačtena. Pokud běžíte lokálně, můžete tuto buňku přeskočit.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Při trénování velkých modelů může být přidělování paměti na GPU problémem. Také může být potřeba experimentovat s různými velikostmi minibatch, aby se data vešla do paměti GPU, a přitom bylo trénování dostatečně rychlé. Pokud spouštíte tento kód na vlastním GPU stroji, můžete experimentovat s úpravou velikosti minibatch pro zrychlení trénování.

> **Poznámka**: Některé verze ovladačů NVidia jsou známy tím, že po trénování modelu neuvolňují paměť. V tomto notebooku spouštíme několik příkladů, a to může v určitých uspořádáních způsobit vyčerpání paměti, zvláště pokud provádíte vlastní experimenty v rámci stejného notebooku. Pokud narazíte na nějaké podivné chyby při spuštění trénování modelu, můžete chtít restartovat jádro notebooku.


In [3]:
batch_size = 16
embed_size = 64

## Jednoduchý RNN klasifikátor

V případě jednoduché RNN je každá rekurentní jednotka jednoduchá lineární síť, která přijímá vektor vstupu a vektor stavu a produkuje nový vektor stavu. V Kerasu to může být reprezentováno vrstvou `SimpleRNN`.

Ačkoliv můžeme do RNN vrstvy přímo předat tokeny s one-hot kódováním, není to dobrý nápad kvůli jejich vysoké dimenzionalitě. Proto použijeme vrstvu embedding, která sníží dimenzionalitu vektorů slov, následovanou RNN vrstvou a nakonec klasifikátorem `Dense`.

> **Poznámka**: V případech, kdy dimenzionalita není tak vysoká, například při použití tokenizace na úrovni znaků, může dávat smysl předat one-hot kódované tokeny přímo do RNN buňky.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Poznámka:** Pro jednoduchost zde používáme neučenou vrstvu embeddingu, ale pro lepší výsledky můžeme použít předtrénovanou vrstvu embeddingu pomocí Word2Vec, jak bylo popsáno v předchozí části. Bylo by dobré cvičení, abyste tento kód upravili tak, aby fungoval s předtrénovanými embeddingy.

Nyní si natrénujeme naši RNN. RNN jsou obecně docela obtížné na natrénování, protože jakmile jsou RNN buňky rozvinuty podél délky sekvence, výsledný počet vrstev zapojených do zpětné propagace je poměrně velký. Proto musíme zvolit menší rychlost učení a trénovat síť na větším datasetu, abychom dosáhli dobrých výsledků. To může trvat poměrně dlouho, proto je preferováno použití GPU.

Pro zrychlení budeme trénovat model RNN pouze na titulcích zpráv, vynecháme popis. Můžete zkusit trénovat s popisem a uvidíte, jestli se vám podaří model natrénovat.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Poznámka**: Přesnost zde pravděpodobně bude nižší, protože trénujeme pouze na titulcích zpráv.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Nyní, když používáme maskování, můžeme trénovat model na celém datasetu názvů a popisů.

> **Poznámka**: Všimli jste si, že jsme používali vektorový transformátor natrénovaný na titulech zpráv, a ne na celém obsahu článku? Potenciálně to může způsobit, že některé tokeny budou ignorovány, takže je lepší vektorový transformátor znovu natrénovat. Nicméně to může mít jen velmi malý efekt, takže pro jednoduchost zůstaneme u předchozího předtrénovaného vektorového transformátoru.


## LSTM: Dlouhá krátkodobá paměť

Jedním z hlavních problémů RNN jsou **mizící gradienty**. RNN mohou být poměrně dlouhé a mohou mít problém s propagací gradientů zpět až k první vrstvě sítě během zpětné propagace. Když se to stane, síť nemůže naučit vztahy mezi vzdálenými tokeny. Jedním ze způsobů, jak tomuto problému předejít, je zavést **explicitní správu stavu** pomocí **bran**. Dvě nejběžnější architektury, které zavádějí brány, jsou **long short-term memory** (LSTM) a **gated relay unit** (GRU). Zde si popíšeme LSTM.

![Obrázek ukazující příklad buňky dlouhé krátkodobé paměti](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Síť LSTM je uspořádána podobně jako RNN, ale jsou zde dva stavy, které se předávají z vrstvy na vrstvu: vlastní stav $c$ a skrytý vektor $h$. V každé jednotce je skrytý vektor $h_{t-1}$ kombinován se vstupem $x_t$ a společně řídí, co se stane se stavem $c_t$ a výstupem $h_{t}$ pomocí **bran**. Každá brána má sigmoidní aktivaci (výstup v rozsahu $[0,1]$), kterou lze považovat za bitovou masku při násobení se stavovým vektorem. LSTM má následující brány (zleva doprava na obrázku výše):
* **zapomínací brána**, která určuje, které složky vektoru $c_{t-1}$ je třeba zapomenout a které předat dál.
* **vstupní brána**, která určuje, kolik informací ze vstupního a předchozího skrytého vektoru by mělo být začleněno do stavového vektoru.
* **výstupní brána**, která vezme nový stavový vektor a rozhodne, které jeho složky budou použity k vytvoření nového skrytého vektoru $h_t$.

Složky stavu $c$ lze považovat za příznaky, které lze zapínat a vypínat. Například když narazíme na jméno *Alice* v posloupnosti, odhadneme, že se jedná o ženu, a zvýšíme příznak ve stavu, který říká, že ve větě máme ženské podstatné jméno. Když pak dále narazíme na slova *and Tom*, zvýšíme příznak, že máme množné číslo podstatného jména. Manipulací se stavem tak můžeme sledovat gramatické vlastnosti věty.

> **Poznámka**: Zde je skvělý zdroj pro pochopení interní struktury LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) od Christophera Olaha.

Ačkoliv vnitřní struktura buňky LSTM může vypadat složitě, Keras skrývá tuto implementaci uvnitř vrstvy `LSTM`, takže v příkladu výše stačí pouze nahradit rekurentní vrstvu:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Poznámka** že trénování LSTM je také pomalé a na začátku tréninku nemusíte zaznamenat výrazný nárůst přesnosti. Možná budete muset pokračovat v tréninku nějakou dobu, abyste dosáhli dobré přesnosti.


## Obousměrné a vícevrstvé RNN

V našich dosavadních příkladech rekurentní sítě pracují od začátku sekvence až do jejího konce. Připadá nám to přirozené, protože to odpovídá směru, kterým čteme nebo posloucháme řeč. Nicméně v případech, kdy je potřeba náhodný přístup k vstupní sekvenci, dává větší smysl provádět rekurentní výpočet v obou směrech. RNN, které umožňují výpočty v obou směrech, se nazývají **obousměrné** RNN a lze je vytvořit obalením rekurentní vrstvy speciální vrstvou `Bidirectonal`.

> **Poznámka**: Vrstva `Bidirectional` vytvoří dvě kopie vrstvy uvnitř sebe a nastaví vlastnost `go_backwards` jedné z těchto kopií na `True`, čímž ji spustí v opačném směru podél sekvence.

Rekurentní sítě, ať už jednosměrné nebo obousměrné, zachycují vzory v sekvenci a ukládají je do stavových vektorů nebo je vracejí jako výstup. Stejně jako u konvolučních sítí můžeme za první vrstvu přidat další rekurentní vrstvu, která zachytí vzory na vyšší úrovni, sestavené z nižších vzorů extrahovaných první vrstvou. Tím se dostáváme k pojmu **vícevrstvé RNN**, které se skládá ze dvou nebo více rekurentních sítí, kde výstup předchozí vrstvy je předáván jako vstup vrstvě následující.

![Obrázek znázorňující vícevrstvé dlouhodobé krátkodobé paměťové RNN](../../../../../translated_images/cs/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Obrázek z [tohoto úžasného příspěvku](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) od Fernanda Lópeze.*

Keras usnadňuje konstrukci těchto sítí, protože stačí do modelu přidat další rekurentní vrstvy. U všech vrstev kromě poslední musíme specifikovat parametr `return_sequences=True`, protože potřebujeme, aby vrstva vracela všechny mezilehlé stavy, ne pouze konečný stav rekurentního výpočtu.

Postavme si dvouvrstvé obousměrné LSTM pro náš klasifikační problém.

> **Poznámka** tento kód opět trvá poměrně dlouho na dokončení, ale poskytuje nám dosud nejvyšší přesnost. Takže možná stojí za to počkat a výsledky sledovat.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN pro jiné úkoly

Doposud jsme se zaměřili na použití RNN pro klasifikaci sekvencí textu. Ale zvládnou mnohem více úkolů, například generování textu a strojový překlad &mdash; tyto úkoly si probereme v další části.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
